# 🧪 Laboratorio Interactivo: Agente Emma (OfficeFlow Supply Co.)

¡Bienvenido a este cuaderno de experimentación interactiva! Este notebook complementa el curso **Building Reliable Agents** de LangChain Academy.

Aquí podrás:
1. **Ver el grafo de arquitectura** y entender el flujo de decisión del agente.
2. **Inspeccionar la base de datos real SQLite** (`inventory.db`) con `pandas`.
3. **Probar a Emma celda por celda** utilizando tu servidor local de **Ollama** (`qwen2.5:7b` + `nomic-embed-text`) en tu GPU **NVIDIA RTX 3060** a coste $0.
4. **Analizar las llamadas a herramientas y trazas** registradas en tu proyecto de **LangSmith**.

## 1. 🗺️ Arquitectura del Grafo y Nodos de Decisión (Emma v1)

Emma no es un simple chatbot de una sola pasada; opera mediante una arquitectura de **bucle de herramientas (Tool-Calling Loop)** que puede modelarse formalmente como un grafo de estados cíclico:

```mermaid
flowchart TD
    Inicio([👤 Usuario envía pregunta]) --> NodoInit["1. Nodo Contexto<br/>(System Prompt + Memoria Thread)"]
    NodoInit --> NodoLLM1["2. Nodo Razonamiento - LLM<br/>(Evalúa pregunta frente a herramientas)"]
    NodoLLM1 --> Decision{"¿Requiere herramientas?"}
    Decision -- "SÍ (tool_calls detectado)" --> NodoTools["3. Nodo Despachador de Tools"]
    
    subgraph Herramientas ["🧰 Caja de Herramientas (Tools)"]
        NodoTools -->|Consulta Catálogo o Stock| ToolSQL["query_database<br/>(SQLite: inventory.db)"]
        NodoTools -->|Políticas Corporativas| ToolRAG["search_knowledge_base<br/>(RAG: nomic-embed-text)"]
    end
    
    ToolSQL --> NodoRetorno["4. Nodo Formateo de Resultados<br/>(Añade mensaje con role: tool)"]
    ToolRAG --> NodoRetorno
    NodoRetorno --> NodoLLM2["5. Nodo Síntesis - LLM<br/>(Interpreta datos y redacta respuesta)"]
    NodoLLM2 --> Decision2{"¿Requiere otra herramienta?"}
    Decision2 -- "SÍ" --> NodoTools
    Decision2 -- "NO" --> NodoFin["6. Nodo Entrega y Memoria<br/>(Guarda en historial y retorna)"]
    Decision -- "NO (charla general)" --> NodoFin
    NodoFin --> Fin([💬 Respuesta final entregada al usuario])
```


In [ ]:
# Renderizado gráfico en alta definición (PNG / SVG compilado con Mermaid)
import os
from IPython.display import Image, display

img_candidates = [
    os.path.abspath("architecture_graph.png"),
    os.path.abspath(os.path.join("..", "architecture_graph.png")),
]
for p in img_candidates:
    if os.path.exists(p):
        display(Image(filename=p, width=820))
        break
else:
    print("Nota: Visualiza el diagrama Mermaid renderizado en la celda superior.")


### 📋 Mapeo Detallado de Nodos del Grafo

| Nodo | Nombre | Función en `agent_v1.py` | Responsabilidad y Flujo |
| :--- | :--- | :--- | :--- |
| **1** | **Contexto e Historial** | `get_thread_history()` + `system_prompt` | Recupera los mensajes previos de la sesión (`thread_id`) y fusiona el System Prompt de Emma con la nueva pregunta del usuario. |
| **2** | **Razonamiento LLM** | `client.chat.completions.create()` | El LLM (`qwen2.5:7b`) analiza el contexto y decide con `tool_choice="auto"` si puede responder directamente o si emite uno o más `tool_calls`. |
| **3** | **Despachador de Tools** | `while response_message.tool_calls:` | Itera sobre cada llamada a herramienta solicitada por el modelo y la despacha a su función ejecutora correspondiente. |
| **4** | **Formateo de Resultados** | `messages.append({"role": "tool", ...})` | Empaqueta la salida de cada tool con el estándar de OpenAI (`role: tool`, `tool_call_id`) para que el LLM sepa qué resultado corresponde a cada llamada. |
| **5** | **Síntesis y Re-evaluación** | `client.chat.completions.create()` | El LLM recibe la respuesta de la tool y evalúa si necesita ejecutar otra herramienta o si ya puede sintetizar la respuesta final en lenguaje natural. |
| **6** | **Entrega y Persistencia** | `save_thread_history()` | Guarda todos los intercambios en el almacén de memoria (`thread_store`) y entrega la respuesta final al usuario. |

---

### 🧰 Las Herramientas del Agente (Tools)

Emma v1 cuenta con dos herramientas especializadas declaradas mediante esquemas JSON de Function Calling:

#### 1. `query_database` (Inventario SQL)
- **Objetivo**: Ejecutar consultas SQL en la base de datos `inventory/inventory.db` (SQLite).
- **Esquema JSON expuesto al LLM**:
  ```json
  {
    "name": "query_database",
    "description": "SQL query to get information about our inventory for customers like products, quantities and prices.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {"type": "string", "description": "SQL query to execute against the inventory database"}
      },
      "required": ["query"]
    }
  }
  ```
- ⚠️ **Punto Crítico / Falencia de Emma v1**: El prompt de la herramienta le dice que consulte `prices` (precios), pero en la base de datos **no existe la columna `price`**. Como Emma v1 no tiene un paso de introspección de esquema (`PRAGMA table_info`), a veces adivina nombres de tablas y columnas, causando errores SQL. Esto se corrige en la Lección 4 con `agent_v2`.

#### 2. `search_knowledge_base` (RAG Semántico de Políticas)
- **Objetivo**: Búsqueda por similitud semántica en documentos Markdown (`knowledge_base/documents/`).
- **Motor Vectorial**: Calcula similitud coseno entre el embedding de la pregunta y los embeddings cacheados (`nomic-embed-text` de 768 dimensiones).
- **Top-K**: Recupera los 2 fragmentos (`top_k=2`) con mayor puntuación de similitud.
- ⚠️ **Punto Crítico / Falencia de Emma v1**: Emma v1 divide los documentos en *chunks* usando encabezados Markdown. A veces un párrafo relevante queda separado de su contexto y el Top-2 no recupera la política completa. Esto se corrige en `agent_v4` pasando a documentos completos sin fragmentar (*no-chunking*).


In [ ]:
# Inspección interactiva del código del bucle del grafo (Tool-Calling Loop)
import inspect
import agent_v1

# Mostramos el bucle de control del grafo extraído directamente de agent_v1
source_lines = inspect.getsourcelines(agent_v1.chat)[0]
# Filtramos desde la llamada inicial al LLM hasta el fin del bucle de herramientas
bucle_codigo = "".join(source_lines[14:68])
print("=== BUCLE DE CONTROL DEL GRAFO (Tool-Calling Loop en agent_v1.py) ===\n")
print(bucle_codigo)

## 2. ⚙️ Configuración del Entorno y Verificación de Modelos

Cargamos las variables de entorno desde el archivo `.env`.

In [4]:
from dotenv import load_dotenv

# Cargamos el archivo .env de la raíz del curso
load_dotenv(os.path.abspath(os.path.join("..", ".env")))

base_url = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")
chat_model = os.getenv("CHAT_MODEL", "qwen2.5:7b")
embed_model = os.getenv("EMBEDDING_MODEL", "nomic-embed-text")
langsmith_project = os.getenv("LANGSMITH_PROJECT", "lca-reliable-agents")
langsmith_tracing = os.getenv("LANGSMITH_TRACING", "false")

print("=== ESTADO DEL ENTORNO ===")
print(f"📡 Endpoint API:         {base_url}")
print(f"🧠 Modelo de Chat:        {chat_model}")
print(f"📐 Modelo de Embeddings:  {embed_model}")
print(f"📊 Proyecto LangSmith:    {langsmith_project}")
print(f"🔍 Trazabilidad Activa:   {langsmith_tracing}")

=== ESTADO DEL ENTORNO ===
📡 Endpoint API:         http://localhost:11434/v1
🧠 Modelo de Chat:        qwen2.5:7b
📐 Modelo de Embeddings:  nomic-embed-text
📊 Proyecto LangSmith:    lca-reliable-agents
🔍 Trazabilidad Activa:   true


## 3. 🗄️ Inspección Visual de la Base de Datos (`inventory.db`)

Veamos con qué datos reales cuenta Emma en el almacén de OfficeFlow.

In [6]:
import sqlite3
import pandas as pd

db_path = os.path.abspath(os.path.join("inventory", "inventory.db"))
conn = sqlite3.connect(db_path)

# Consultamos los productos y sus existencias combinando ambas tablas
query = """
SELECT 
    i.item_id AS 'ID',
    i.sku_label AS 'Descripción del Producto',
    s.available_units AS 'Unidades Disponibles'
FROM items i
LEFT JOIN stock_levels s ON i.item_id = s.item_id
ORDER BY s.available_units DESC;
"""

df_inventario = pd.read_sql_query(query, conn)
conn.close()

print(f"Total de artículos registrados en inventario: {len(df_inventario)}\n")
df_inventario.head(10)

Total de artículos registrados en inventario: 5



,ID,Descripción del Producto,Unidades Disponibles
0,1,Copy Paper 500 Sheets,45
1,4,Spiral Notebooks (3-pack),31
2,2,Blue Ballpoint Pens (12-pack),23
3,3,Stapler with Staples,12
4,5,Manila File Folders (25-pack),8


## 4. 🚀 Inicializar el Agente y Cargar la Base de Conocimiento

Cargamos los documentos de políticas corporativas en memoria.

In [7]:
import agent_v1
from agent_v1 import chat, load_knowledge_base

# Cargamos la base de conocimiento (usa la caché local embeddings_nomic...json)
await load_knowledge_base()
print("\n✅ Agente Emma listo para interactuar con soporte de LangSmith Tracing.")

Knowledge base loaded from cache (embeddings_nomic-embed-text.json): 127 chunks

✅ Agente Emma listo para interactuar con soporte de LangSmith Tracing.


## 5. 💬 Prueba 1: Consulta de Productos y Base de Datos SQL

Le preguntaremos a Emma sobre artículos en stock. Observaremos si decide usar `query_database`.

In [ ]:
from IPython.display import Markdown

pregunta_1 = "Hola Emma, ¿qué tipos de bolígrafos tienen y qué disponibilidad tienen en almacén?"
print(f"Tú: {pregunta_1}\n")

# Invocamos al agente
resultado_1 = await chat(pregunta_1)

# Mostramos la respuesta final generada
display(Markdown(f"**Emma:** {resultado_1['output']}"))

### 🔍 Inspección Anatómica de la Respuesta
Veamos los mensajes internos que ocurrieron entre bambalinas (llamadas a herramientas y respuestas):

In [ ]:
print(f"Thread ID de la conversación: {agent_v1.thread_id}\n")

for i, msg in enumerate(resultado_1["messages"]):
    rol = msg.get("role", "")
    if rol == "system":
        continue  # Omitimos el system prompt largo por legibilidad
    print(f"--- [Paso {i}] Rol: {rol.upper()} ---")
    if "tool_calls" in msg and msg["tool_calls"]:
        for tc in msg["tool_calls"]:
            print(f"  🛠️ Invocó Tool: {tc['function']['name']}")
            print(f"  📥 Argumentos: {tc['function']['arguments']}")
    elif rol == "tool":
        print(f"  📤 Salida de Tool: {msg.get('content')}")
    else:
        contenido = msg.get("content", "")
        print(f"  💬 Texto: {contenido[:150]}..." if len(contenido) > 150 else f"  💬 Texto: {contenido}")
    print()

## 6. 📚 Prueba 2: Consulta de Políticas Corporativas (RAG)

Ahora haremos una pregunta sobre devoluciones para forzar el uso de `search_knowledge_base`.

In [ ]:
pregunta_2 = "¿Cuál es la política de devoluciones de la empresa y cuántos días tengo para devolver un producto?"
print(f"Tú: {pregunta_2}\n")

resultado_2 = await chat(pregunta_2)
display(Markdown(f"**Emma:** {resultado_2['output']}"))

## 7. 🧪 Laboratorio Libre: Haz tu propia pregunta a Emma

Modifica la variable `mi_pregunta` y corre la celda:

In [ ]:
# Escribe aquí lo que quieras probar:
mi_pregunta = "¿Tienen papel de copia para entrega inmediata?"

mi_resultado = await chat(mi_pregunta)
display(Markdown(f"**Emma:** {mi_resultado['output']}"))

## 8. 📊 Visualización en LangSmith Tracing

Todas las consultas que acabas de ejecutar en este notebook se registraron en tu panel de LangSmith con el identificador de sesión activo:

1. Ingresa a [https://smith.langchain.com/](https://smith.langchain.com/).
2. Selecciona el proyecto: **`lca-reliable-agents`**.
3. En la barra de filtros, busca:
   `metadata.thread_id == "..."` (el ID impreso arriba).
4. Observa el árbol de ejecución jerárquico (*Run Tree*), los tokens consumidos y las latencias de cada llamada a herramientas.

---